# PA4 Task 3 — mechanism diagnostic (supplementary, ~75 min on 2 × T4)

Runs **after** the main Task 3 run and changes no main-run condition. It tests three explanations of the main results
(`scripts/diagnose_task3.py`, `src/diag.py`):

| Phase | GPU 0 | GPU 1 |
|---|---|---|
| 1 | **split**: the split-prefill gap in FP16 vs FP32, and where it sits (~5 min) | **math**: greedy GSM8K accuracy of the target, the original and the four merged drafts, plus draft entropy on fixed and own prefixes (~60 min) |
| 2 (alone, for clean timing) | **launch**: decode-step time vs layers kept and vs batch size, profiler busy share (~10 min) | — |

Outputs: `results/diagnostics/`, tables D1–D3 and figure FD1, zipped as `task3_diagnostics.zip`.

**Before running**
1. The main run's `PA4/Task 3/results/` must be committed and pushed.
2. Upload `task3_adapters.zip` (from the main run) as a Kaggle dataset and add it to this notebook
   (**Add Input → Your Datasets**, or the **Upload** button in the Input panel).
3. Settings: **GPU T4 × 2**, **Internet on**.

In [ ]:
REPO_URL = "https://github.com/wasaymj/ATML.git"
BRANCH   = "main"

import os, subprocess, sys, shutil, json, glob, time, zipfile
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
TASK = f"{REPO}/PA4/Task 3"
DATA = f"{REPO}/PA4/atml-pa4-datasets"
LOGS = f"{WORK}/logs"
ADAPT = f"{WORK}/t3_adapters"
NL = chr(10)
os.makedirs(LOGS, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
need = ["results/banks/teacher.json.gz", "results/banks/student_on_policy_s6305.json.gz", "results/quality/original.json.gz",
        "results/distill/off_policy_s6304/presentations.jsonl.gz", "scripts/diagnose_task3.py"]
missing = [p for p in need if not os.path.exists(f"{TASK}/{p}")]
assert not missing, f"commit and push the main Task 3 results and the diagnostic code first: {missing}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"], check=True)

In [ ]:
# adapters: a Kaggle dataset made from task3_adapters.zip (auto-extracted or still zipped)
os.makedirs(ADAPT, exist_ok=True)
names = ["off_policy_s6304", "off_policy_s6305", "on_policy_s6304", "on_policy_s6305"]
for cfgf in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True):
    d = os.path.dirname(cfgf)
    if os.path.basename(d) in names:
        shutil.copytree(d, f"{ADAPT}/{os.path.basename(d)}", dirs_exist_ok=True)
if not all(os.path.exists(f"{ADAPT}/{n}/adapter_config.json") for n in names):
    for z in glob.glob("/kaggle/input/**/task3_adapters*.zip", recursive=True):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(WORK + "/_adapters_zip")
    for n in names:
        src = glob.glob(f"{WORK}/_adapters_zip/**/{n}/adapter_config.json", recursive=True)
        if src:
            shutil.copytree(os.path.dirname(src[0]), f"{ADAPT}/{n}", dirs_exist_ok=True)
have = [n for n in names if os.path.exists(f"{ADAPT}/{n}/adapter_config.json")]
print("adapters:", have)
assert len(have) == 4, "add task3_adapters.zip as a Kaggle dataset (Add Input) and rerun this cell"

In [ ]:
rev = json.load(open(f"{DATA}/data/local_source_revisions.json"))["revisions"]
from huggingface_hub import snapshot_download
for repo_id, key in [("Qwen/Qwen2.5-1.5B-Instruct", "model_target"), ("Qwen/Qwen2.5-0.5B-Instruct", "model_draft")]:
    print(repo_id, rev[key], snapshot_download(repo_id, revision=rev[key]))
ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false",
           HF_HUB_DISABLE_PROGRESS_BARS="1")
r = subprocess.run([sys.executable, "scripts/run_task3.py", "--step", "data", "--results", f"{WORK}/_data_step"], cwd=TASK, env=ENV)
assert r.returncode == 0, "GSM8K data step failed"

In [ ]:
def launch(name, gpu, args):
    script = f"{LOGS}/{name}.sh"
    cmd = f'{sys.executable} scripts/diagnose_task3.py {args} || echo "STEP FAILED"'
    open(script, "w").write(f'cd "{TASK}"' + NL + cmd + NL)
    print(f"[{name}] GPU {gpu}: {args}", flush=True)
    return subprocess.Popen(["bash", script], stdout=open(f"{LOGS}/{name}.log", "a"), stderr=subprocess.STDOUT,
                            env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))

def wait(procs, every_min=5):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every_min * 60)
        for name in procs:
            tail = open(f"{LOGS}/{name}.log", errors="replace").read().splitlines()[-3:]
            print(f"--- {name} ({time.strftime('%H:%M')}):", *tail, sep=NL + "   ", flush=True)
    print({n: p.returncode for n, p in procs.items()})

import torch
two = torch.cuda.device_count() >= 2
# phase 1: split (GPU 0) and math (GPU 1) in parallel
if two:
    wait({"split": launch("split", 0, "--part split"), "math": launch("math", 1, f'--part math --adapters "{ADAPT}"')})
else:
    wait({"split": launch("split", 0, "--part split")})
    wait({"math": launch("math", 0, f'--part math --adapters "{ADAPT}"')})
# phase 2: launch-bound timing with nothing else running
wait({"launch": launch("launch", 0, "--part launch")})
for f in glob.glob(f"{LOGS}/*.log"):
    print(f, "<-- contains a failed step" if "STEP FAILED" in open(f, errors="replace").read() else "ok")

In [ ]:
rc = subprocess.run([sys.executable, "scripts/compile_task3_report.py"], cwd=TASK, env=ENV).returncode
print("report exit code", rc)
shutil.copytree(LOGS, f"{TASK}/results/diagnostics/logs", dirs_exist_ok=True)
out = f"{WORK}/task3_diagnostics"
shutil.rmtree(out, ignore_errors=True)
shutil.copytree(f"{TASK}/results/diagnostics", f"{out}/results/diagnostics")
os.makedirs(f"{out}/results/tables", exist_ok=True); os.makedirs(f"{out}/results/figures", exist_ok=True)
for f in glob.glob(f"{TASK}/results/tables/D*_*"):
    shutil.copy(f, f"{out}/results/tables/")
for f in glob.glob(f"{TASK}/results/figures/FD*_*"):
    shutil.copy(f, f"{out}/results/figures/")
shutil.make_archive(f"{WORK}/task3_diagnostics", "zip", out, "results")
print(sorted(os.listdir(WORK)))

In [ ]:
from IPython.display import Markdown, Image, display
for t in sorted(glob.glob(f"{TASK}/results/tables/D*.md")):
    display(Markdown(f"### {os.path.basename(t)}" + NL + open(t, encoding="utf-8").read()))
for fig in sorted(glob.glob(f"{TASK}/results/figures/FD*.png")):
    display(Image(fig, width=900))